# Segment density comparison over the last 1.5 us

Independent density analysis for `alhx_700`, `alhx_800`, and `alhx_1000`, following the Hel50 algorithm in `Analysis/rdf/rdf.ipynb`: segment atom masses are binned by distance to the protein center of mass and converted to mg/mL.

In [ ]:
# Independent analysis: segment density profiles from the last 1.5 us

import os
import sys
from pathlib import Path

import MDAnalysis as mda
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator
from tqdm import tqdm


# =========================
# User settings
# =========================
BASE_DIR = Path('/data/jychen/MD_projects/curvature_phase/Alpha-Synuclein/martini3001/Martini_alhx/mdrun/M3IDP')
W_CLUSTER_DIR = BASE_DIR / 'W_cluster'
OUT_DIR = BASE_DIR / 'Analysis/EN_modeltest/diff_k_density'
OUT_DIR.mkdir(parents=True, exist_ok=True)
IDP_BASELINE_FILE = BASE_DIR / 'Analysis/rdf/segment_density_IDP_v2.txt'

# MDAnalysis reads GROMACS trajectory time in ps. 1.5 us = 1,500,000 ps.
LAST_WINDOW_US = 1.5
LAST_WINDOW_PS = LAST_WINDOW_US * 1_000_000.0

STRIDE = 1
RESIDUES_PER_CHAIN = 140
TARGET_SEGMENTS = {
    'N-ter (1-60)': (1, 60),
    'NAC (61-95)': (61, 95),
    'C-term (96-140)': (96, 140),
}

# Same density settings as the Hel50 cell in Analysis/rdf/rdf.ipynb.
R_MAX_A = 150.0
N_BINS = 15
CONVERSION_FACTOR = 1660.54
REUSE_EXISTING_DATA = True

FIG_PRESET = 'double_short'
ARIAL_TTF_PATH = '/data/jychen/package/Arial/arial.ttf'
USE_ARIAL_TTF = True

SYSTEMS = {
    '700': {
        'label': r'Hel50($k=700$)',
        'color': '#d62728',
        'topology': W_CLUSTER_DIR / 'alhx_700/TR_Mdvwhole/pbc.tpr',
        'trajectories': [
            W_CLUSTER_DIR / 'alhx_700/TR_Mdvwhole/clus_centered.xtc',
            W_CLUSTER_DIR / 'alhx_700_re1/TR_Mdvwhole/clus_centered.xtc',
            W_CLUSTER_DIR / 'alhx_700_re2/TR_Mdvwhole/clus_centered.xtc',
        ],
    },
    '800': {
        'label': r'Hel50($k=800$)',
        'color': '#2ca02c',
        'topology': W_CLUSTER_DIR / 'alhx_800/TR_mdvwhole/pbc.tpr',
        'trajectories': [
            W_CLUSTER_DIR / 'alhx_800/TR_mdvwhole/clus_centered.xtc',
            W_CLUSTER_DIR / 'alhx_800_re1/TR_mdvwhole/clus_centered.xtc',
        ],
    },
    '1000': {
        'label': r'Hel50($k=1000$)',
        'color': '#9467bd',
        'topology': W_CLUSTER_DIR / 'alhx_1000/TR_mdvwhole/pbc.tpr',
        'trajectories': [
            W_CLUSTER_DIR / 'alhx_1000/TR_mdvwhole/clus_centered.xtc',
            W_CLUSTER_DIR / 'alhx_1000_re1/TR_mdvwhole/clus_centered.xtc',
        ],
    },
}


# =========================
# Publication figure style
# =========================
_PUB_FONT_NAME = None

def load_pub_font():
    global _PUB_FONT_NAME
    if _PUB_FONT_NAME is not None:
        return _PUB_FONT_NAME
    if USE_ARIAL_TTF and os.path.exists(ARIAL_TTF_PATH):
        fm.fontManager.addfont(ARIAL_TTF_PATH)
        _PUB_FONT_NAME = fm.FontProperties(fname=ARIAL_TTF_PATH).get_name()
    else:
        _PUB_FONT_NAME = 'Arial'
    print(f'Using font: {_PUB_FONT_NAME}')
    return _PUB_FONT_NAME

def mm_to_inch(mm):
    return mm / 25.4

FIG_SIZE = {'single': 90, 'double': 180}
FIG_PRESETS = {
    'single_small': ('single', 60),
    'single_short': ('single', 75),
    'double_short': ('double', 70),
    'double_medium': ('double', 100),
    'double_large': ('double', 140),
}

def set_pub_style():
    pub_font = load_pub_font()
    mpl.rcParams.update({
        'font.family': pub_font,
        'font.sans-serif': [pub_font, 'Arial', 'Helvetica', 'Liberation Sans', 'DejaVu Sans'],
        'font.weight': 'normal',
        'axes.labelweight': 'normal',
        'axes.titleweight': 'normal',
        'font.size': 9,
        'axes.labelsize': 10,
        'axes.titlesize': 10,
        'xtick.labelsize': 9,
        'ytick.labelsize': 9,
        'legend.fontsize': 9,
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'svg.fonttype': 'none',
        'axes.linewidth': 1.5,
        'xtick.major.width': 1.4,
        'ytick.major.width': 1.4,
        'xtick.major.size': 4.5,
        'ytick.major.size': 4.5,
        'xtick.direction': 'in',
        'ytick.direction': 'in',
        'lines.linewidth': 1.7,
        'legend.frameon': False,
        'axes.unicode_minus': False,
        'axes.spines.top': True,
        'axes.spines.right': True,
        'savefig.dpi': 600,
        'figure.dpi': 120,
    })

def new_subplots(nrows, ncols, preset=FIG_PRESET, **kwargs):
    set_pub_style()
    mode, height_mm = FIG_PRESETS[preset]
    return plt.subplots(
        nrows,
        ncols,
        figsize=(mm_to_inch(FIG_SIZE[mode]), mm_to_inch(height_mm)),
        **kwargs,
    )

def savefig_pub(fig, filename, dpi=600, tight=True):
    if tight:
        fig.savefig(filename, dpi=dpi, bbox_inches='tight', pad_inches=0.02)
    else:
        fig.savefig(filename, dpi=dpi)


# =========================
# Density helpers
# =========================
bins = np.linspace(0, R_MAX_A, N_BINS + 1)
bin_centers_a = 0.5 * (bins[:-1] + bins[1:])
bin_centers_nm = bin_centers_a / 10.0
shell_volumes_a3 = 4.0 / 3.0 * np.pi * (bins[1:] ** 3 - bins[:-1] ** 3)

def get_last_window_start_frame(u, last_window_ps):
    n_frames = len(u.trajectory)
    first_time = float(u.trajectory[0].time)
    last_time = float(u.trajectory[-1].time)
    start_time = last_time - last_window_ps

    if start_time <= first_time:
        print(
            f'  Trajectory length is {(last_time - first_time) / 1_000_000.0:.3f} us; '
            f'use all {n_frames} frames.'
        )
        return 0, first_time, last_time

    for frame_idx, ts in enumerate(u.trajectory):
        if float(ts.time) >= start_time:
            print(
                f'  Use last {last_window_ps / 1_000_000.0:.3f} us: '
                f'time {ts.time / 1_000_000.0:.3f}-{last_time / 1_000_000.0:.3f} us, '
                f'frames {frame_idx}-{n_frames - 1}.'
            )
            return frame_idx, float(ts.time), last_time

    return max(0, n_frames - 1), last_time, last_time

def build_segment_atomgroups(u):
    protein_atoms = u.select_atoms('protein')
    all_res = protein_atoms.residues
    n_total_res = len(all_res)
    if n_total_res % RESIDUES_PER_CHAIN != 0:
        raise ValueError(
            f'Protein residue count ({n_total_res}) is not divisible by '
            f'RESIDUES_PER_CHAIN ({RESIDUES_PER_CHAIN}).'
        )
    n_chains = n_total_res // RESIDUES_PER_CHAIN
    print(f'  Detected {n_chains} protein chains by residue count.')

    seg_info = {}
    for name, (start_res, end_res) in TARGET_SEGMENTS.items():
        combined_atoms = mda.AtomGroup([], u)
        for chain_idx in range(n_chains):
            chain_offset = chain_idx * RESIDUES_PER_CHAIN
            idx_start = chain_offset + (start_res - 1)
            idx_end = chain_offset + end_res
            combined_atoms += all_res[idx_start:idx_end].atoms
        seg_info[name] = {
            'atoms': combined_atoms,
            'masses': combined_atoms.masses,
            'counts': np.zeros(N_BINS, dtype=float),
        }
    return protein_atoms, seg_info

def analyze_density_trajectory(topology, traj_file):
    u = mda.Universe(str(topology), str(traj_file))
    start_frame, start_time_ps, last_time_ps = get_last_window_start_frame(u, LAST_WINDOW_PS)
    protein_atoms, seg_info = build_segment_atomgroups(u)

    frame_count = 0
    for ts in tqdm(u.trajectory[start_frame::STRIDE], desc='density', unit='frame'):
        current_origin = protein_atoms.center_of_mass()
        for name in TARGET_SEGMENTS:
            atoms = seg_info[name]['atoms']
            masses = seg_info[name]['masses']
            distances = np.linalg.norm(atoms.positions - current_origin, axis=1)
            hist, _ = np.histogram(distances, bins=bins, weights=masses)
            seg_info[name]['counts'] += hist
        frame_count += 1

    if frame_count == 0:
        raise RuntimeError(f'No frames selected for {traj_file}')

    profiles = {}
    for name in TARGET_SEGMENTS:
        avg_mass_in_bins = seg_info[name]['counts'] / frame_count
        density_da_a3 = avg_mass_in_bins / shell_volumes_a3
        profiles[name] = density_da_a3 * CONVERSION_FACTOR

    traj_info = {
        'trajectory': str(traj_file),
        'n_total_frames': len(u.trajectory),
        'start_frame': start_frame,
        'start_time_ps': start_time_ps,
        'last_time_ps': last_time_ps,
        'n_analyzed_frames': frame_count,
    }
    return profiles, traj_info

def analyze_and_cache_system(system_id, cfg):
    profile_file = OUT_DIR / f'segment_density_alhx_{system_id}_last_{LAST_WINDOW_US:g}us.csv'
    replicate_file = OUT_DIR / f'segment_density_replicates_alhx_{system_id}_last_{LAST_WINDOW_US:g}us.csv'
    info_file = OUT_DIR / f'trajectory_window_alhx_{system_id}_last_{LAST_WINDOW_US:g}us.csv'

    if REUSE_EXISTING_DATA and profile_file.exists():
        print(f'alhx_{system_id}: loaded cached profile from {profile_file}')
        return pd.read_csv(profile_file)

    missing = [path for path in [cfg['topology'], *cfg['trajectories']] if not path.exists()]
    if missing:
        raise FileNotFoundError('Missing input files:\n' + '\n'.join(str(path) for path in missing))

    replicate_profiles = []
    traj_rows = []
    print('=' * 70)
    print(f'alhx_{system_id}: analyzing {len(cfg["trajectories"])} trajectories')
    print('=' * 70)

    for rep_idx, traj_file in enumerate(cfg['trajectories'], start=1):
        print(f'\n--- alhx_{system_id} trajectory {rep_idx}/{len(cfg["trajectories"])} ---')
        print(f'  {traj_file}')
        profiles, traj_info = analyze_density_trajectory(cfg['topology'], traj_file)
        traj_info.update({'system': f'alhx_{system_id}', 'replicate': rep_idx})
        traj_rows.append(traj_info)

        rep_df = pd.DataFrame({'radius_nm': bin_centers_nm})
        rep_df['system'] = f'alhx_{system_id}'
        rep_df['replicate'] = rep_idx
        for name, density in profiles.items():
            rep_df[name] = density
        rep_df['Total Protein'] = sum(profiles.values())
        replicate_profiles.append(rep_df)

    all_rep_df = pd.concat(replicate_profiles, ignore_index=True)
    all_rep_df.to_csv(replicate_file, index=False)
    pd.DataFrame(traj_rows).to_csv(info_file, index=False)

    density_cols = list(TARGET_SEGMENTS) + ['Total Protein']
    mean_df = all_rep_df.groupby('radius_nm', as_index=False)[density_cols].mean()
    mean_df.insert(1, 'system', f'alhx_{system_id}')
    mean_df.to_csv(profile_file, index=False)
    print(f'alhx_{system_id}: saved mean profile to {profile_file}')
    print(f'alhx_{system_id}: saved replicate profiles to {replicate_file}')
    print(f'alhx_{system_id}: saved trajectory-window info to {info_file}')
    return mean_df


# =========================
# Run analysis and plot
# =========================
profile_by_system = {
    system_id: analyze_and_cache_system(system_id, cfg)
    for system_id, cfg in SYSTEMS.items()
}
all_profiles = pd.concat(profile_by_system.values(), ignore_index=True)
all_profiles_file = OUT_DIR / f'segment_density_profiles_alhx_700_800_1000_last_{LAST_WINDOW_US:g}us.csv'
all_profiles.to_csv(all_profiles_file, index=False)

plot_col = 'Total Protein'
fig, ax = new_subplots(1, 1, preset='single_short')

idp_baseline = np.atleast_2d(np.loadtxt(IDP_BASELINE_FILE))
idp_radius_nm = idp_baseline[:, 0]
idp_total_density = np.sum(idp_baseline[:, 1:], axis=1)
ax.plot(
    idp_radius_nm,
    idp_total_density,
    color='black',
    lw=mpl.rcParams['lines.linewidth'],
    ls='--',
    label=r'IDP50',
)

for system_id, cfg in SYSTEMS.items():
    df = profile_by_system[system_id]
    ax.plot(
        df['radius_nm'],
        df[plot_col],
        color=cfg['color'],
        lw=mpl.rcParams['lines.linewidth'],
        label=cfg['label'],
    )

ax.set_xlabel('Radius (nm)')
ax.set_ylabel('Total protein density (mg/mL)')
ax.set_xlim(0, R_MAX_A / 10.0)
ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
ax.tick_params(top=False, right=False)
ax.grid(False)
ax.legend(frameon=False, loc='upper right', handlelength=1.8)
fig.tight_layout(pad=0.8)

out_png = OUT_DIR / f'total_density_IDP_700_800_1000_last_{LAST_WINDOW_US:g}us.png'
out_pdf = OUT_DIR / f'total_density_IDP_700_800_1000_last_{LAST_WINDOW_US:g}us.pdf'
savefig_pub(fig, out_png, tight=True)
savefig_pub(fig, out_pdf, tight=True)

summary_rows = [{
    'system': 'IDP50_baseline',
    'Total Protein_peak_mg_ml': float(idp_total_density.max()),
    'Total Protein_integral_mg_ml_nm': float(np.trapz(idp_total_density, idp_radius_nm)),
}]
for system_id, df in profile_by_system.items():
    summary_rows.append({
        'system': f'alhx_{system_id}',
        'Total Protein_peak_mg_ml': float(df[plot_col].max()),
        'Total Protein_integral_mg_ml_nm': float(np.trapz(df[plot_col], df['radius_nm'])),
    })
summary = pd.DataFrame(summary_rows).set_index('system')
summary_file = OUT_DIR / f'total_density_summary_IDP_700_800_1000_last_{LAST_WINDOW_US:g}us.csv'
summary.to_csv(summary_file)

print(summary.round(3))
print(f'Saved combined profiles: {all_profiles_file}')
print(f'Saved summary: {summary_file}')
print(f'Saved figure: {out_png}')
print(f'Saved figure: {out_pdf}')

plt.show()



